In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Create a Gen AI Agent Evaluation for a Deployed Agent

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/create_genai_agent_evaluation.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fgemini%2Fevaluation%2Fcreate_genai_agent_evaluation.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/gemini/evaluation/create_genai_agent_evaluation.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo"><br> Open in Vertex AI Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/create_genai_agent_evaluation.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/create_genai_agent_evaluation.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/create_genai_agent_evaluation.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/create_genai_agent_evaluation.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/create_genai_agent_evaluation.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/create_genai_agent_evaluation.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>

| Author(s) |
| --- |
| [Kelsi Lakey](https://github.com/lakeyk) |
| [Bo Zheng](https://github.com/coolalexzb) |

## Overview

This Colab notebook demonstrates how to use the Gen AI Eval SDK to evaluate a deployed Agent.

- **Run Agent Inference:** First run agent inference to retrieve real responses and traces from the deployed agent.
- **Create Evaluation Run:** Then create an Evaluation Run to perform the Gen AI Agent Evaluation. This Evaluation will be persisted and accessible later.

If you do not have a deployed Agent, please see:
- [Create & Deploy Agent and Run Gen AI Agent Evaluation]()

## Get started

### Install Google Gen AI SDK and other required packages
Restart runtime after installation to load latest packages


In [ ]:
%pip install --upgrade --force-reinstall -q 'google-cloud-aiplatform[evaluation]'
%pip install --upgrade -q openai

### Set Google Cloud project information

To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [ ]:
import vertexai
from google.cloud import storage
from google.genai import types as genai_types
from vertexai import Client

def get_or_create_gcs_bucket(project_id: str, gcs_dest: str) -> str:
    """Retrieves GCS bucket or creates a default."""
    full_path = gcs_dest or f"{project_id}/agent-evaluation"
    path_no_prefix = full_path.removeprefix("gs://")
    bucket_name = path_no_prefix.split("/")[0]

    storage_client = storage.Client(project=project_id)
    if storage_client.lookup_bucket(bucket_name) is None:
        print(f"Creating bucket: {bucket_name}")
        storage_client.create_bucket(bucket_name)

    return f"gs://{path_no_prefix}"


# Configuration
# fmt: off
PROJECT_ID = 'sa-learning-1' # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
LOCATION = 'us-east1' # @param {type: "string", placeholder: "us-central1", isTemplate: true}
GCS_DEST = "gs://sa-learning-1-agent-engine"  # @param {type: "string", placeholder: "[your-gcs-bucket]", isTemplate: true}
# fmt: on
GCS_DEST = get_or_create_gcs_bucket(PROJECT_ID, GCS_DEST)
AGENT = 'projects/sa-learning-1/locations/us-east1/reasoningEngines/932596966586580992' # @param {type: "string", placeholder: "[your-agent]", isTemplate: true}

# Initialize SDK
vertexai.init(project=PROJECT_ID, location=LOCATION)

client = Client(
    project=PROJECT_ID,
    location=LOCATION,
)

### Import libraries


In [ ]:
import time

import pandas as pd
from vertexai import types

# Step 1: Prepare Agent Dataset

## Define Agent Info
You will need to manually update the below `AgentInfo` definition with the input data specific to your agent. \
An example definition is provided below.

In [ ]:
# Define agent functions
get_customer_context = genai_types.FunctionDeclaration(
    description="Fetches holistic customer data including persona, dietary tags, price sensitivity, loyalty tier, recent purchase history, and current market context (weather/season/promotions).",
    name="get_customer_context",
    parameters={
        "properties": {
            "customer_id": {"type": "string", "description": "The unique ID of the customer (e.g., 'CUST001')."}
        },
        "required": ["customer_id"],
        "type": "object",
    },
)

generate_seo_keywords = genai_types.FunctionDeclaration(
    description="Uses an LLM to generate 3 SEO-optimized keywords for a blog post based on the theme, customer persona, and dietary preferences.",
    name="generate_seo_keywords",
    parameters={
        "properties": {
            "blog_theme": {"type": "string", "description": "The topic or theme of the blog post."},
            "persona_summary": {"type": "string", "description": "Summary of the customer persona."},
            "dietary_tags": {"type": "array", "items": {"type": "string"}, "description": "Customer dietary tags (e.g., 'vegan', 'bio-only')."},
            "language": {"type": "string", "description": "Output language for keywords (default 'English')."},
        },
        "required": ["blog_theme", "persona_summary", "dietary_tags"],
        "type": "object",
    },
)

get_product_recommendations = genai_types.FunctionDeclaration(
    description="Article-aware product recommendations with category diversity and LLM curation. Embeds the blog_theme via VECTOR_SEARCH to find article-relevant products, enforces category diversity (max 3 per category), and curates 4-5 best products via LLM.",
    name="get_product_recommendations",
    parameters={
        "properties": {
            "customer_id": {"type": "string", "description": "The unique ID of the customer."},
            "blog_theme": {"type": "string", "description": "The blog post topic — drives product selection."},
            "language": {"type": "string", "description": "Language for usage_tips (default 'English')."},
        },
        "required": ["customer_id"],
        "type": "object",
    },
)

generate_blog_image = genai_types.FunctionDeclaration(
    description="Generates an AI marketing image for a blog post using Gemini image generation and saves it as an ADK artifact. The image is rendered inline automatically.",
    name="generate_blog_image",
    parameters={
        "properties": {
            "prompt": {"type": "string", "description": "A vivid description of the hero image for the blog post."},
            "aspect_ratio": {"type": "string", "description": "Output aspect ratio (default '16:9')."},
            "image_size": {"type": "string", "description": "Output resolution (default '2K')."},
        },
        "required": ["prompt"],
        "type": "object",
    },
)

# Define agent info
agent_info = types.evals.AgentInfo(
    agents={
        "rewe_marketing_agent": types.evals.AgentConfig(
            agent_id="rewe_marketing_agent",
            instruction="You are the REWE Marketing Expert Agent. Your mission is to generate personalized, SEO-optimized blog posts that drive product sales while maintaining the REWE brand voice. Follow a strict workflow: identify customer, gather context, generate SEO keywords, write a blog post (~300 words), generate a hero image, provide article-relevant product recommendations, and verify dietary safety. Ensure language consistency throughout. Use REWE brand tone: professional, friendly, food-obsessed.",
            tools=[
                genai_types.Tool(
                    function_declarations=[
                        get_customer_context,
                        generate_seo_keywords,
                        get_product_recommendations,
                        generate_blog_image,
                    ]
                )
            ],
        )
    },
    root_agent_id="rewe_marketing_agent",
)

## Define Agent Dataset
Define a dataset that is specific to your agent. \
`agent_prompts` should consist of prompts or requests to be made to your agent. A few example prompts are shown below. \
`session_inputs` are required for traces. For more information see [Session](https://google.github.io/adk-docs/sessions/session/).

In [ ]:
session_inputs = types.evals.SessionInput(
    user_id="user_123",
    state={},
)

agent_prompts = [
    "Generate a blogpost for CUST001 on Summer Berry Season",
    "Generate a blogpost for CUST061 on Summer Barbecue Ideas"
]

agent_dataset = pd.DataFrame(
    {
        "prompt": agent_prompts,
        "session_inputs": [session_inputs] * len(agent_prompts),
    }
)

# Step 2: Run Agent Inference

Run inference using your deployed agent. This will add `intermediate_events` and `response` columns to your dataset to be evaluated in the next step.

In [ ]:
# Custom inference: bypasses eval SDK's broken stream_query() by using
# streaming_agent_run_with_events, which works correctly over gRPC.
import asyncio, json as _json, nest_asyncio
from vertexai.agent_engines import AgentEngine
nest_asyncio.apply()

agent_engine = AgentEngine(AGENT)
prompt_df = agent_dataset.copy() if isinstance(agent_dataset, __import__("pandas").DataFrame) else agent_dataset.eval_dataset_df.copy()

resp_col, int_ev_col, ad_col = [], [], []

for idx, row in prompt_df.iterrows():
    print(f'Prompt {idx+1}/{len(prompt_df)}', end=' ')
    # Parse session_inputs
    si = row.get('session_inputs')
    if isinstance(si, str): si = _json.loads(si)
    elif hasattr(si, 'model_dump'): si = si.model_dump()
    elif not isinstance(si, dict): si = {}
    user_id = si.get('user_id', 'eval_user')

    # Create session
    try:
        sess = agent_engine.create_session(user_id=user_id, state=si.get('state', {}))
        session_id = sess.get('id','') if isinstance(sess, dict) else str(sess)
    except Exception as e:
        print(f'session error: {e}')
        resp_col.append(_json.dumps({'error': str(e)}))
        int_ev_col.append([]); ad_col.append(None); continue

    prompt_text = str(row.get('prompt', ''))
    req = _json.dumps({'user_id': user_id, 'session_id': session_id,
        'message': {'role': 'user', 'parts': [{'text': prompt_text}]}})

    # Stream events
    all_events = []
    try:
        async def _collect():
            evs = []
            async for chunk in agent_engine.streaming_agent_run_with_events(request_json=req):
                if isinstance(chunk, dict):
                    for ev in chunk.get('events', []):
                        if isinstance(ev, dict): evs.append(ev)
            return evs
        all_events = asyncio.get_event_loop().run_until_complete(_collect())
    except Exception as e:
        print(f'run error: {e}')
        resp_col.append(_json.dumps({'error': str(e)}))
        int_ev_col.append([]); ad_col.append(None); continue

    # Extract final text from last event with text
    final_text = None
    for ev in reversed(all_events):
        c = ev.get('content')
        if not isinstance(c, dict): continue
        for p in c.get('parts', []):
            if not isinstance(p, dict): continue
            if 'function_call' in p or 'function_response' in p: continue
            if p.get('thought') and 'text' not in p: continue
            if 'text' in p and p['text'].strip():
                final_text = p['text']; break
        if final_text: break

    resp_col.append(final_text or _json.dumps({'error': 'No text in events'}))
    print(f'=> {len(all_events)} events, resp={len(final_text or "")} chars')

    # Intermediate events (all except last)
    int_ev_col.append([{'event_id': e.get('id'), 'content': e.get('content'),
        'creation_timestamp': e.get('timestamp'), 'author': e.get('author')}
        for e in all_events[:-1]])

    # Agent data
    try:
        agent_events = [types.evals.AgentEvent(author=e.get('author','model'),
            content=genai_types.Content.model_validate(e['content']))
            for e in all_events if isinstance(e.get('content'), dict)]
        turn = types.evals.ConversationTurn(turn_index=0, turn_id='turn_0', events=agent_events)
        ad_col.append(types.evals.AgentData(turns=[turn]).model_dump(exclude_unset=True))
    except Exception:
        ad_col.append(None)

prompt_df['response'] = resp_col
prompt_df['intermediate_events'] = int_ev_col
prompt_df['agent_data'] = ad_col

agent_dataset_with_inference = types.EvaluationDataset(
    eval_dataset_df=prompt_df, candidate_name='agent_engine_0')
print(f'\n✅ Done. Columns: {list(prompt_df.columns)}')
agent_dataset_with_inference.eval_dataset_df


Prompt 1/2 => 9 events, resp=2728 chars
Prompt 2/2 => 7 events, resp=3393 chars

✅ Done. Columns: ['prompt', 'session_inputs', 'response', 'intermediate_events', 'agent_data']


,prompt,session_inputs,response,intermediate_events,agent_data
0,Generate a blogpost for CUST001 on Summer Berr...,user_id='user_123' state={} app_name=None,*The blog hero image has been successfully gen...,[{'event_id': 'fe934b11-d6a7-4ca5-ad5f-05d300b...,"{'turns': [{'turn_index': 0, 'turn_id': 'turn_..."
1,Generate a blogpost for CUST061 on Summer Barb...,user_id='user_123' state={} app_name=None,I have successfully generated your blog post a...,[{'event_id': '4666441f-c5f4-47be-a080-fec6940...,"{'turns': [{'turn_index': 0, 'turn_id': 'turn_..."


# Step 3: Run Gen AI Agent Evaluation

Run Gen AI Agent Evaluation using the Evaluation Management Service. \
This will persist your dataset and evaluation results which can be retrieved via the Agent Engine UI.

In [ ]:
evaluation_run = client.evals.create_evaluation_run(
    dataset=agent_dataset_with_inference,
    agent=AGENT,
    agent_info=agent_info,
    metrics=[
        types.RubricMetric.FINAL_RESPONSE_QUALITY,
        types.RubricMetric.TOOL_USE_QUALITY,
        types.RubricMetric.HALLUCINATION,
        types.RubricMetric.SAFETY,
    ],
    dest=GCS_DEST,
)
# Display the Evaluation Run status and results
evaluation_run.show()

/tmp/ipykernel_3624900/2798841815.py:1: ExperimentalWarning: The Vertex SDK GenAI evals.create_evaluation_run module is experimental, and may change in future versions.
  evaluation_run = client.evals.create_evaluation_run(
/usr/local/lib/python3.12/dist-packages/vertexai/_genai/_evals_common.py:2642: ExperimentalWarning: The Vertex SDK GenAI evals.create_evaluation_item module is experimental, and may change in future versions.
  candidate_names: list[str],
/usr/local/lib/python3.12/dist-packages/vertexai/_genai/_evals_common.py:2649: ExperimentalWarning: The Vertex SDK GenAI evals.create_evaluation_set module is experimental, and may change in future versions.
  and candidate_names
/usr/local/lib/python3.12/dist-packages/vertexai/_genai/_evals_common.py:366: ExperimentalWarning: The Vertex SDK GenAI evals.get_evaluation_set method is experimental, and may change in future versions.
  return last_event.content, all_events[:-1]
/usr/local/lib/python3.12/dist-packages/vertexai/_genai/_e

### Poll Evaluation Run for Completion and Display Results
Retrieve the Evaluation Run and directly display the results using the .show() command. If the Evaluation Run failed the error message will be displayed. Otherwise the following results data will be displayed in an embedded report.

- **Summary metrics:** An aggregated view of all metrics, showing the mean score and standard deviation across the entire dataset.
- **Agent info:** Information describing the evaluated agent, including developer instruction, agent description, tool definitions, etc. Applied for agent evaluation only.
- **Detailed results:** A case-by-case breakdown, allowing you to inspect the prompt, reference, candidate response, and the specific score and explanation for each metric. For agent evaluation, detailed results will also include traces showing the agent interactions.

For more information, see [Visualizing Evaluation Reports](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/models/view-evaluation#visualizing-evaluation-reports).

In [ ]:
while evaluation_run.state not in {"SUCCEEDED", "FAILED", "CANCELLED"}:
    evaluation_run.show()
    evaluation_run = client.evals.get_evaluation_run(name=evaluation_run.name)
    time.sleep(5)


evaluation_run = client.evals.get_evaluation_run(
    name=evaluation_run.name, include_evaluation_items=True
)

# Display the Evaluation Run status and results
evaluation_run.show()

/tmp/ipykernel_3598778/1049331569.py:3: ExperimentalWarning: The Vertex SDK GenAI evals.get_evaluation_run module is experimental, and may change in future versions.
  evaluation_run = client.evals.get_evaluation_run(name=evaluation_run.name)
